# nano Beatrix — the identity read: is a path head's cost the path, or the change of rotation?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pillar1h.ipynb)

**The question**: The group read (`nano_pillar1g.ipynb`) switched the path off one block at a time by putting that block's path head on the block's own rotation: the head holding the lookup carried most of the cost, and each other path head still cost a little, though it hardly attends to the earlier copies of its path. That switch does two things at once: it removes the path's matches, and it gives the head a positional rotation it was never trained with (away from its matches a path head meets its keys at unrelated phases, so it sees no position). This notebook separates the two: each path head is put instead on fresh random phases, its own draw for the query and the key at every position. No path match is left, and the head stays position-free. Nothing trains.

| models | on random phases | reads |
|---|---|---|
| the path in blocks 1-3, seeds 1-6 | no block, each of blocks 1, 2, 3 alone, all three | 30 |
| the path in every block, seeds 1-6 | no block, each block alone, all six | 48 |

For each block, the cost of the random phases is what the path's matches carry; the group read's cost minus it is what the change of rotation does. With no block on random phases each read equals the model's stored text read, bit for bit: the notebook checks it before it ships anything.

**The cells**: C1 pins the trainer and holds the decision block; C2 writes the held-out file and checks the twelve models, their stored text reads and the group read's records on [AbstractPhil/geolip-rotary-experiments](https://huggingface.co/AbstractPhil/geolip-rotary-experiments); C3 reads each model (a model whose record is already on the Hub is skipped) and ships its record to `bed/reports/group/` (`<run>_group_random.json`); C4 renders the table beside the group read's.

**Before C1**: Runtime > Change runtime type > the G4 card. The notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on, for the records' upload. Run C1, then Runtime > Restart session and run all. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/ninth_head.py`: `random_phases`, `random_blocks`; `rotary_scale.py`: `block_read(mode="random")`; `tests/test_identity_read.py`).


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.97" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 97), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU (the stored reads were taken in bf16 on the card)'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the shipper reads it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out file, the models, the records
REPO = 'AbstractPhil/geolip-rotary-experiments'      # the models, their stored reads and the group read's records are here; the records ship here
TAIL = lambda L: f'_L{L}_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
PIECE = {'early': 'nhpn16b123', 'every': 'nhpn16'}     # the name's piece: the path in blocks 1-3's ninth heads; in every block's
SERIES = {'early': 'pillar1c', 'every': 'pillar1b'}    # their series
TAG = lambda a, s: f'alhena-both_s{s}{TAIL(24000).replace("_askany_", "_askany_p75d_")}_cp4_gc48_kh50_va1_pt10b4h0_krt16r218_{PIECE[a]}_ahdigit1_{SERIES[a]}'
SETS = {'early': [[], [1], [2], [3], [1, 2, 3]],       # no block, each path block alone, all three (five reads)
        'every': None}                                 # the default: no block, each block alone, all six (eight reads)
MODE = 'random'                                        # the named blocks on fresh random phases (no path match, still position-free)
LABEL = {'early': 'the path in blocks 1-3', 'every': 'the path in every block'}
MODELS = [(f'{LABEL[a]}, seed {s}', a, s, TAG(a, s)) for a in ('early', 'every') for s in range(1, 7)]
TEXT_DOCS, TEXT_MB = 128, 16                           # the stored text reads: the trainer's 128 held-out documents in batches of 16
GROUP_DIR = 'bed/reports/group'                        # this notebook's records (series pillar1h) and the group read's (pillar1g)
TEXT_DIR = 'bed/reports/text'                          # the stored text reads (the parity anchor)
print('decision block:', len(MODELS), 'models |', {a: (len(v) if v else 'no block, each alone, all') for a, v in SETS.items()}, '| mode', MODE,
      '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')


In [ ]:
# C2 — THE DATA AND THE MODELS: the held-out head (the trainer's 2,048 reserved fineweb-edu rows), and the twelve models with their
# weights, their ninth heads, their stored text reads and the group read's records on the Hub.
import json, os, time
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from geolip.nano_beatrix import bed_corpus
from geolip.nano_beatrix import rotary_scale as rs
HELD = bed_corpus.write_heldout(f'{OUT}/bed')
print('held-out head', os.path.basename(HELD), round(os.path.getsize(HELD) / 1e6, 1), 'MB')
api = HfApi(token=HF_TOKEN)
hub = set(api.list_repo_files(REPO))
need = lambda t: (f'bed/weights/{t}/model.safetensors', f'bed/weights/{t}/extra_head.safetensors', f'bed/weights/{t}/scale.json',
                  f'{TEXT_DIR}/{rs.text_name(t)}.json', f'{GROUP_DIR}/{rs.group_name(t)}.json')
missing = [lab for lab, a, s, t in MODELS if not all(f in hub for f in need(t))]
assert not missing, ('models, their stored reads or the group read records are not on the Hub', missing)
before = [lab for lab, a, s, t in MODELS if f'{GROUP_DIR}/{rs.group_name(t, MODE)}.json' in hub]
print(f'the {len(MODELS)} models with their text reads and group records on {REPO}: {len(MODELS)} of {len(MODELS)}; records there before this session: {len(before)}')


In [ ]:
# C3 — THE READS: each model fetched from the Hub and read with the named path blocks on random phases (rotary_scale.block_read,
# mode random), the read with no block changed checked against the model's stored text read bit for bit, the record shipped to
# bed/reports/group/ (a background shipper with retries). A model whose record is on the Hub or in OUT/group/ is not read again.
import shutil
lship = rs.Shipper(REPO) if HF_TOKEN else None
os.makedirs(f'{OUT}/group', exist_ok=True)
todo = [m for m in MODELS if f'{GROUP_DIR}/{rs.group_name(m[3], MODE)}.json' not in hub]
t0 = time.time()
for i, (lab, a, s, tag) in enumerate(todo):
    out = f'{OUT}/group/{rs.group_name(tag, MODE)}.json'
    if not os.path.exists(out):                       # a record made here before a restart: no download, its check again
        snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN)
    rec = rs.block_read(f'{OUT}/w/bed/weights/{tag}', out, HELD, sets=SETS[a], n_docs=TEXT_DOCS, mb=TEXT_MB, dev='cuda', log=lambda m: None,
                        mode=MODE)
    tr = json.load(open(hf_hub_download(REPO, f'{TEXT_DIR}/{rs.text_name(tag)}.json', local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
    by = {tuple(r['off']): r for r in rec['reads']}
    if not (by[()]['bits'] == tr['bits'] and by[()]['by_repeat'] == tr['by_repeat'] and rec.get('mode') == MODE):
        os.replace(out, out + '.bad')                  # kept for the look, never shipped
        raise AssertionError((lab, 'the read with no block changed is not the stored text read: the instrument first', by[()]['bits'], tr['bits']))
    if lship:
        lship.submit(f'{OUT}/group', GROUP_DIR)
    shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)   # the session's disk holds one model at a time
    el = time.time() - t0
    print(f"{lab}: no block changed {by[()]['bits']:.4f}; " + ', '.join(f"random {'+'.join(map(str, o))} {r['bits'] - by[()]['bits']:+.4f}"
          for o, r in by.items() if o) + f" | {i + 1} of {len(todo)} read, {el / 60:.1f} min spent, about {el / (i + 1) * (len(todo) - i - 1) / 60:.1f} min left", flush=True)
if lship:
    print('the records shipped' if lship.drain() else 'the records still retrying in the background')


In [ ]:
# C4 — THE TABLE (any time; the records made here or read from the Hub): for each model and path block, the cost of random phases
# (what the path's matches carry), the group read's cost of the block's own rotation, and their difference (what the change of
# rotation does).
from IPython.display import Markdown, display

def _rec(local, hub_path):
    if os.path.exists(local):
        return json.load(open(local, encoding='utf-8'))
    try:
        return json.load(open(hf_hub_download(REPO, hub_path, local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
    except Exception:
        return None

for a in ('early', 'every'):
    rows, head = [], None
    for lab, a2, s, tag in MODELS:
        if a2 != a:
            continue
        r = _rec(f'{OUT}/group/{rs.group_name(tag, MODE)}.json', f'{GROUP_DIR}/{rs.group_name(tag, MODE)}.json')
        g = _rec(f'{OUT}/hub/{GROUP_DIR}/{rs.group_name(tag)}.json', f'{GROUP_DIR}/{rs.group_name(tag)}.json')
        if r is None or g is None:
            continue
        rb = {tuple(x['off']): x['bits'] for x in r['reads']}
        gb = {tuple(x['off']): x['bits'] for x in g['reads']}
        sets = [o for o in rb if o and o in gb]
        if head is None:
            head = ['| seed | ' + ' | '.join(f"block{'s' if len(o) > 1 else ''} {'+'.join(map(str, o))}: random / own rotation" for o in sets) + ' |',
                    '|---' * (len(sets) + 1) + '|']
        rows.append(f'| {s} | ' + ' | '.join(f'{rb[o] - rb[()]:+.4f} / {gb[o] - gb[()]:+.4f}' for o in sets) + ' |')
    if rows:
        display(Markdown(f'**{LABEL[a]}** (held-out bits a byte against the model as trained)\n\n' + '\n'.join(head + rows)))
